# 25 - Procesamiento de imagenes con K-Means: color, reconstruccion y superficies

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender que es una imagen digital en terminos matematicos.
2. Leer una imagen en Python y analizar su estructura (`shape`, `dtype`, rangos).
3. Interpretar canales de color RGB y su papel en la representacion visual.
4. Aplicar K-Means para reducir la cantidad de colores (cuantizacion).
5. Reconstruir una imagen a partir de una paleta reducida.
6. Conectar la segmentacion por color con estimacion de superficies.
7. Identificar errores comunes al trabajar con imagenes y clustering.


## Ruta de la sesion

1. Como funciona una imagen: pixeles, canales y matriz.
2. Lectura de imagen en Python.
3. Exploracion de colores y canales.
4. K-Means para cuantizacion de color.
5. Reconstruccion con menos colores y analisis de perdida.
6. Segmentacion por color para estimar superficies.
7. Errores comunes.
8. Ejercicios de pensamiento.


## 1) Como funciona una imagen digital

Una imagen es una matriz de pixeles.

Para una imagen RGB, cada pixel tiene 3 numeros:

- canal **R** (rojo),
- canal **G** (verde),
- canal **B** (azul).

Si el arreglo tiene forma `(alto, ancho, 3)`, entonces:

1. `alto` = numero de filas,
2. `ancho` = numero de columnas,
3. `3` = numero de canales.

En imagenes `uint8`, cada canal va de 0 a 255.
Asi, cada color es un punto en un espacio 3D (R, G, B).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image


In [ ]:
# Mini ejemplo artificial: 2 filas, 3 columnas, 3 canales (RGB)
imagen_mini = np.array(
    [
        [[255, 0, 0], [0, 255, 0], [0, 0, 255]],
        [[255, 255, 0], [255, 255, 255], [0, 0, 0]],
    ],
    dtype=np.uint8,
)

print("Shape imagen_mini:", imagen_mini.shape)
print("Pixel [0,0] (rojo):", imagen_mini[0, 0])
print("Pixel [1,1] (blanco):", imagen_mini[1, 1])

plt.figure(figsize=(4, 2.5))
plt.imshow(imagen_mini)
plt.title("Imagen mini RGB")
plt.axis("off")
plt.show()


## 2) Lectura de imagen en Python

Usaremos una imagen de ejemplo incluida en `scikit-learn` para evitar dependencias externas.
Despues puedes reemplazarla por cualquier imagen local.


In [ ]:
imagen = load_sample_image("china.jpg")

print("Tipo:", type(imagen))
print("Shape:", imagen.shape)
print("Dtype:", imagen.dtype)
print("Min/Max:", imagen.min(), imagen.max())


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(imagen)
plt.title("Imagen original")
plt.axis("off")
plt.show()


## 3) Canales de color y distribucion

Separar canales ayuda a entender que informacion aporta cada componente de color.
Un mismo objeto puede verse muy distinto si observamos solo R, G o B.


In [ ]:
canal_r = imagen[:, :, 0]
canal_g = imagen[:, :, 1]
canal_b = imagen[:, :, 2]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(canal_r, cmap="Reds")
axes[0].set_title("Canal R")
axes[0].axis("off")

axes[1].imshow(canal_g, cmap="Greens")
axes[1].set_title("Canal G")
axes[1].axis("off")

axes[2].imshow(canal_b, cmap="Blues")
axes[2].set_title("Canal B")
axes[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(canal_r.ravel(), bins=30, color="red", alpha=0.7)
axes[0].set_title("Histograma R")
axes[0].set_xlabel("Intensidad")

axes[1].hist(canal_g.ravel(), bins=30, color="green", alpha=0.7)
axes[1].set_title("Histograma G")
axes[1].set_xlabel("Intensidad")

axes[2].hist(canal_b.ravel(), bins=30, color="blue", alpha=0.7)
axes[2].set_title("Histograma B")
axes[2].set_xlabel("Intensidad")

plt.tight_layout()
plt.show()


## 4) Idea clave: cuantizacion de color con K-Means

K-Means agrupa puntos similares. Aqui, cada pixel es un punto en RGB.

Proceso:

1. Convertimos la imagen `(H, W, 3)` a una tabla `(H*W, 3)`.
2. Ejecutamos K-Means para encontrar `k` colores representativos (centroides).
3. Cada pixel se reemplaza por el centroide de su cluster.
4. Reconstruimos la imagen con forma original.

Efecto: menos colores, menor complejidad visual y compresion aproximada.


In [ ]:
def cuantizar_colores(img: np.ndarray, k: int, random_state: int = 42):
    h, w, c = img.shape
    pixeles = img.reshape(-1, c).astype(np.float64)

    modelo = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    etiquetas = modelo.fit_predict(pixeles)

    paleta = np.clip(modelo.cluster_centers_, 0, 255).astype(np.uint8)
    reconstruida = paleta[etiquetas].reshape(h, w, c)
    etiquetas_2d = etiquetas.reshape(h, w)

    return reconstruida, etiquetas_2d, paleta, modelo


In [ ]:
k_values = [64, 16, 8, 4]
resultados = {}

for k in k_values:
    resultados[k] = cuantizar_colores(imagen, k)

fig, axes = plt.subplots(1, len(k_values) + 1, figsize=(18, 5))

axes[0].imshow(imagen)
axes[0].set_title("Original")
axes[0].axis("off")

for idx, k in enumerate(k_values, start=1):
    img_k, _, _, _ = resultados[k]
    axes[idx].imshow(img_k)
    axes[idx].set_title(f"k = {k}")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
h, w, _ = imagen.shape
bytes_original = h * w * 3  # uint8 por canal

print("Tamano aproximado original (bytes):", bytes_original)

for k in [64, 16, 8, 4]:
    bits_por_etiqueta = int(np.ceil(np.log2(k)))
    bytes_etiquetas = (h * w * bits_por_etiqueta) / 8
    bytes_paleta = k * 3
    total_aprox = bytes_etiquetas + bytes_paleta
    ratio = total_aprox / bytes_original
    print(f"k={k:>2} -> bytes aprox: {total_aprox:,.0f}, ratio: {ratio:.3f}")


In [ ]:
def mse(img_a: np.ndarray, img_b: np.ndarray) -> float:
    a = img_a.astype(np.float64)
    b = img_b.astype(np.float64)
    return float(np.mean((a - b) ** 2))


print("Error de reconstruccion (MSE) vs original:")
for k in [64, 16, 8, 4]:
    img_k, _, _, _ = resultados[k]
    print(f"k={k:>2} -> MSE: {mse(imagen, img_k):.2f}")


## 5) De colores a superficies: idea aplicada

Si segmentas una imagen por color, puedes estimar superficie por conteo de pixeles.

Flujo conceptual:

1. Clusterizar pixeles para separar regiones cromaticas.
2. Elegir el cluster de interes (ej. vegetacion, agua, suelo).
3. Contar pixeles del cluster.
4. Convertir pixeles a area real con una escala fisica.

Formula:

$$\text{Area\_objetivo} = \frac{\#\text{pixeles\_objetivo}}{\#\text{pixeles\_totales}} \times \text{Area\_total\_real}$$

La escala fisica puede venir de:

- una regla u objeto de tamano conocido,
- metadatos de dron/satelite,
- calibracion de camara.


In [ ]:
k_superficie = 6
img_k, labels_2d, paleta, _ = cuantizar_colores(imagen, k=k_superficie)

print("Paleta RGB de centroides:")
for i, color in enumerate(paleta):
    print(f"Cluster {i}: {color.tolist()}")

# Heuristica simple: cluster mas verde
indice_verde = int(np.argmax(paleta[:, 1]))
mask_verde = labels_2d == indice_verde

porcentaje_verde = 100 * mask_verde.mean()
print(f"\nCluster seleccionado (mas verde): {indice_verde}")
print(f"Porcentaje de pixeles en ese cluster: {porcentaje_verde:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(imagen)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(img_k)
axes[1].set_title(f"Cuantizada (k={k_superficie})")
axes[1].axis("off")

axes[2].imshow(mask_verde, cmap="gray")
axes[2].set_title("Mascara cluster verde")
axes[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Ejemplo de conversion a area real (suposicion didactica)
area_total_m2 = 12000.0
area_verde_m2 = (porcentaje_verde / 100.0) * area_total_m2

print(f"Si el area total real fuera {area_total_m2:.0f} m2,")
print(f"la superficie estimada del cluster verde seria {area_verde_m2:.1f} m2")


## 6) Validacion con imagen sintetica (area conocida)

Creamos una imagen artificial donde conocemos el area real en pixeles.
Asi podemos verificar si la estimacion por K-Means es razonable.


In [ ]:
h_syn, w_syn = 200, 300
img_syn = np.zeros((h_syn, w_syn, 3), dtype=np.uint8)
img_syn[:, :] = [30, 30, 30]      # fondo oscuro
img_syn[50:150, 60:240] = [220, 40, 40]  # rectangulo rojo

pixeles_rojos_reales = (150 - 50) * (240 - 60)
pixeles_totales = h_syn * w_syn
porc_real = 100 * pixeles_rojos_reales / pixeles_totales

img_syn_k, labels_syn, paleta_syn, _ = cuantizar_colores(img_syn, k=2)
indice_rojo = int(np.argmax(paleta_syn[:, 0]))
mask_roja = labels_syn == indice_rojo
pixeles_rojos_estimados = int(mask_roja.sum())
porc_estimado = 100 * pixeles_rojos_estimados / pixeles_totales

print("Pixeles rojos reales:", pixeles_rojos_reales)
print("Pixeles rojos estimados:", pixeles_rojos_estimados)
print(f"Porcentaje real: {porc_real:.2f}%")
print(f"Porcentaje estimado: {porc_estimado:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_syn)
axes[0].set_title("Imagen sintetica")
axes[0].axis("off")

axes[1].imshow(img_syn_k)
axes[1].set_title("Cuantizada k=2")
axes[1].axis("off")

axes[2].imshow(mask_roja, cmap="gray")
axes[2].set_title("Mascara roja estimada")
axes[2].axis("off")

plt.tight_layout()
plt.show()


## 7) Limitaciones practicas para medir superficies

Aunque la idea funciona, en casos reales hay retos:

1. Cambios de iluminacion alteran colores.
2. Sombras pueden crear clusters falsos.
3. Objetos distintos pueden tener colores similares.
4. Escala fisica mal calibrada produce areas incorrectas.
5. Elegir `k` inadecuado mezcla o fragmenta regiones.

Por eso, en proyectos reales se combina esta tecnica con validacion de campo, reglas de negocio y evaluacion visual.


## 8) Errores comunes (y como evitarlos)

1. **No reescalar forma antes de K-Means**.
   - K-Means espera `(n_samples, n_features)`, no `(H, W, 3)`.

2. **Olvidar reconstruir con la forma original**.
   - Debes usar `.reshape(H, W, 3)` despues de mapear centroides.

3. **Usar `k` muy pequeno sin validar perdida**.
   - Mide calidad con alguna metrica (ej. MSE) y revision visual.

4. **Confundir cluster con clase semantica**.
   - Un cluster es similitud cromatica, no necesariamente objeto real.

5. **Estimar area sin calibracion**.
   - Sin `m2/pixel`, solo tienes porcentajes relativos.


In [ ]:
# Ejemplo correcto de reshape para K-Means y reconstruccion
H, W, C = imagen.shape
pix = imagen.reshape(-1, C)
print("Forma para K-Means:", pix.shape)

k_demo = 5
modelo_demo = KMeans(n_clusters=k_demo, random_state=42, n_init=10)
labels_demo = modelo_demo.fit_predict(pix)
paleta_demo = np.clip(modelo_demo.cluster_centers_, 0, 255).astype(np.uint8)
img_demo = paleta_demo[labels_demo].reshape(H, W, C)

print("Forma reconstruida:", img_demo.shape)


## 9) Ejercicios de pensamiento (no copiar/pegar)

Objetivo: razonar como cientifico/a de datos aplicado/a a vision por computadora.


### Ejercicio 1: elegir `k` con criterio tecnico

Prueba al menos 6 valores de `k`.
Para cada uno, reporta:

1. MSE contra original,
2. impresion visual,
3. costo computacional aproximado.

Concluye que `k` recomendarias para una app movil con recursos limitados.


In [ ]:
# Implementa aqui tu comparacion de k y tu recomendacion final.


### Ejercicio 2: medir una superficie con calibracion

Supone que 1 pixel equivale a 0.04 m2.
Selecciona un cluster de interes y calcula su superficie.

Despues responde:

1. Que tan sensible es el resultado al valor de `k`.
2. Que error esperarias si la calibracion tiene un 10% de sesgo.


In [ ]:
# Calcula aqui el area estimada con tu suposicion de escala.


### Ejercicio 3: robustez ante ruido e iluminacion

Agrega ruido gaussiano o cambia brillo/contraste de la imagen.
Repite la cuantizacion y compara como cambia la mascara de superficie.

Argumenta:

1. que tan estable es tu pipeline,
2. que preprocesamiento agregarias en produccion.


In [ ]:
# Implementa aqui experimento de robustez (ruido / brillo / contraste).


### Ejercicio 4: de notebook a backend

Diseña una API minima para este caso:

1. endpoint para subir imagen,
2. endpoint para ejecutar cuantizacion con `k` configurable,
3. endpoint para devolver porcentaje de superficie objetivo.

Define contrato JSON, validaciones y posibles errores HTTP.


In [ ]:
# Escribe aqui tu diseno de API y ejemplos de request/response.


### Ejercicio 5: limites metodologicos

Redacta una nota tecnica para un equipo no tecnico explicando:

1. cuando este metodo SI es util,
2. cuando NO deberia usarse como unica fuente de decision,
3. que controles minimos exiges antes de desplegarlo.


In [ ]:
# Redacta aqui tu nota tecnica.


## 10) Cierre

Ideas clave:

1. Una imagen RGB es una matriz `(H, W, 3)` de intensidades de color.
2. K-Means puede reducir colores al mapear pixeles a centroides.
3. Menos colores simplifica la imagen, pero puede perder detalle.
4. La segmentacion por color permite estimar superficies por conteo de pixeles.
5. Para uso real, la calibracion y validacion son tan importantes como el algoritmo.
